# Rap-Snacks Boltz Validation v4 — scrambled_na

Folds 111 scrambled native_ala sequences (37 bars × 3 scrambles) through Boltz-2.
Same AA composition as native_ala, random positional order — composition-only baseline.

**Purpose:** separate backbone geometry from composition effect.

| Bucket | Boltz pLDDT | Interpretation |
|--------|-------------|----------------|
| concordance | 0.441 | lyric-derived backbone |
| native_ala | 0.543 | Ala-substituted lyric backbone |
| scrambled_na (this run) | ? | composition baseline |
| free_design | 0.643 | MPNN on concordance backbone |
| native_ala_free | 0.806 | MPNN on native_ala backbone |

**Expected:** scrambled_na ≈ concordance or native_ala — composition alone explains that level; MPNN sequence order matters for foldability above it.

**Input:** `rap_snacks/inputs/outputs/bioreason/scrambled_na_esm.csv` (upload from local `outputs/bioreason/scrambled_na_esm.csv`)

---
## Cell 1 — Install

Run **once** per session. Auto-restarts the runtime after install.  
After restart, **skip this cell** and start from Cell 2.

In [ ]:
import subprocess, sys, os

try:
    import boltz  # noqa
    print('Boltz already installed — skip this cell.')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', 'boltz'], check=True)
    print('Boltz installed. Restarting runtime...')
    os.kill(os.getpid(), 9)

---
## Cell 2 — Mount Drive + Check Inputs

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/rap_snacks')
DRIVE_IN   = DRIVE_ROOT / 'inputs'
DRIVE_RES  = DRIVE_ROOT / 'results'
DRIVE_FIGS = DRIVE_RES  / 'figures'
CONTENT    = Path('/content/scratch')

for p in [DRIVE_IN, DRIVE_RES, DRIVE_FIGS, CONTENT]:
    p.mkdir(parents=True, exist_ok=True)

# Input: scrambled_na_esm.csv (upload from outputs/bioreason/scrambled_na_esm.csv)
SCRAMBLED_NA_CSV = DRIVE_IN / 'outputs' / 'bioreason' / 'scrambled_na_esm.csv'

# v3 results for cross-run comparison
V3_CSV = DRIVE_RES / 'boltz_validation_v3_results.csv'
# v2 results for full picture
V2_CSV = DRIVE_RES / 'boltz_validation_v2_results.csv'

checks = [SCRAMBLED_NA_CSV]
all_ok = True
for p in checks:
    status = '✅' if p.exists() else '❌ MISSING'
    if not p.exists(): all_ok = False
    print(f'{status}  {p}')

for p in [V3_CSV, V2_CSV]:
    status = '✅' if p.exists() else '⚠️  optional'
    print(f'{status}  {p}')

if all_ok:
    print('\nAll required inputs present — ready to run.')
else:
    print('\n⚠️  Upload scrambled_na_esm.csv to Drive:')
    print('  Local path: outputs/bioreason/scrambled_na_esm.csv')
    print('  Drive path: rap_snacks/inputs/outputs/bioreason/scrambled_na_esm.csv')

---
## Cell 3 — Config

In [ ]:
import json, shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

N_MODELS = 1      # 1 diffusion sample per scramble (control run — no need for ensemble)
VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

BOLTZ_IN  = CONTENT / 'boltz_inputs_sc_na'
BOLTZ_OUT = CONTENT / 'boltz_outputs_sc_na'

print(f'N_MODELS={N_MODELS}  (1 model per scramble — composition baseline control)')

---
## Cell 4 — Helpers

In [ ]:
def write_boltz_yaml(name: str, seq: str, out_dir: Path) -> None:
    yaml = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {seq}
      msa: empty
"""
    (out_dir / f'{name}.yaml').write_text(yaml)


def parse_from_drive(drive_out_dir, meta_df):
    """Parse confidence JSONs directly from Drive (no copytree)."""
    preds_dirs = list(drive_out_dir.rglob('predictions'))
    if not preds_dirs:
        raise RuntimeError(f'No predictions/ dir under {drive_out_dir}')
    preds = preds_dirs[0]
    n_json = sum(1 for _ in preds.rglob('confidence_*.json'))
    print(f'Predictions dir: {preds}  ({n_json} JSON files)')

    rows = []
    for _, m in meta_df.iterrows():
        name    = m['name']
        seq_dir = preds / name
        jsons   = sorted(seq_dir.glob('confidence_*.json')) if seq_dir.exists() else []
        if not jsons:
            continue
        plddts, ptms, confs = [], [], []
        for jf in jsons:
            d = json.loads(jf.read_text())
            plddts.append(d.get('complex_plddt', float('nan')))
            ptms.append(d.get('ptm',   float('nan')))
            confs.append(d.get('confidence_score', float('nan')))
        rows.append({
            'name':            name,
            'sequence':        m['sequence'],
            'bar_id':          m['bar_id'],
            'bucket':          m.get('bucket', 'scrambled_na'),
            'scramble_idx':    m.get('scramble_idx', -1),
            'esm_plddt':       m.get('esm_plddt', float('nan')),
            'boltz_plddt':     float(np.nanmean(plddts)),
            'boltz_ptm':       float(np.nanmean(ptms)),
            'boltz_confidence':float(np.nanmean(confs)),
            'n_models':        len(jsons),
        })
    return pd.DataFrame(rows)


print('Helpers defined.')

---
## Cell 5 — Build Boltz Inputs

Loads `scrambled_na_esm.csv` (37 bars × 3 scrambles = 111 seqs) and writes one YAML per scramble.

In [ ]:
BOLTZ_IN.mkdir(parents=True, exist_ok=True)

sc_df = pd.read_csv(SCRAMBLED_NA_CSV)
print(f'Loaded {len(sc_df)} scrambled_na sequences from {SCRAMBLED_NA_CSV.name}')
print(sc_df.head(3).to_string(index=False))

# Parse scramble index from name: bar_6_sc_na_0 → 0
sc_df['scramble_idx'] = sc_df['name'].str.extract(r'_sc_na_(\d+)$').astype(int)

meta_rows = []
n_written = 0
n_skipped = 0

for _, row in sc_df.iterrows():
    seq = str(row['sequence']).upper().strip()
    if not set(seq) <= VALID_AA:
        print(f'  [SKIP] {row["name"]} — invalid AA')
        n_skipped += 1
        continue
    write_boltz_yaml(row['name'], seq, BOLTZ_IN)
    meta_rows.append({
        'name':         row['name'],
        'bar_id':       row['bar_id'],
        'bucket':       'scrambled_na',
        'scramble_idx': row['scramble_idx'],
        'sequence':     seq,
        'esm_plddt':    row.get('esm_plddt', float('nan')),
    })
    n_written += 1

meta_df = pd.DataFrame(meta_rows)
n_yaml  = len(list(BOLTZ_IN.glob('*.yaml')))
print(f'\nWrote {n_written} YAMLs ({n_skipped} skipped)')
print(f'  {BOLTZ_IN}  ({n_yaml} files)')
print(f'\nBars covered: {sorted(meta_df["bar_id"].unique())}')

---
## Cell 6 — Run Boltz

~111 sequences × 1 model each.  
Expected: ~25–35 min on A100.  
Backs up to Drive immediately after completion.

In [ ]:
import subprocess

if BOLTZ_OUT.exists():
    shutil.rmtree(BOLTZ_OUT)
BOLTZ_OUT.mkdir(parents=True, exist_ok=True)

cmd = ['boltz', 'predict', str(BOLTZ_IN),
       '--out_dir', str(BOLTZ_OUT),
       '--diffusion_samples', str(N_MODELS),
       '--output_format', 'pdb',
       '--no_kernels', '--override']
print('Running:', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=False, text=True)

if r.returncode != 0:
    print(f'[ERROR] Boltz failed (exit {r.returncode})')
else:
    print('Boltz complete. Backing up to Drive...')
    drive_out = DRIVE_RES / 'boltz_outputs_sc_na'
    if drive_out.exists():
        shutil.rmtree(drive_out)
    shutil.copytree(BOLTZ_OUT, drive_out)
    n_pdb  = sum(1 for _ in drive_out.rglob('*.pdb'))
    n_json = sum(1 for _ in drive_out.rglob('confidence_*.json'))
    print(f'Backed up → {n_pdb} PDBs, {n_json} confidence JSONs')
    print(f'Drive path: {drive_out}')

---
## Cell 7 — Parse Results

Reads confidence JSONs directly from Drive (no copytree needed).  
If runtime restarted: re-run Cells 2→5 first to rebuild `meta_df`.

In [ ]:
drive_out = DRIVE_RES / 'boltz_outputs_sc_na'
results   = parse_from_drive(drive_out, meta_df)

RESULTS_CSV = DRIVE_RES / 'boltz_validation_v4_scrambled_na_results.csv'
results.to_csv(RESULTS_CSV, index=False)
print(f'Saved {len(results)} rows → {RESULTS_CSV}')

print(f'\nmean Boltz pLDDT (scrambled_na): {results["boltz_plddt"].mean():.3f}')
print(f'sd:                              {results["boltz_plddt"].std():.3f}')
print(f'\n--- Per-bar mean pLDDT ---')
for bar_id, g in results.groupby('bar_id'):
    print(f'  {bar_id:8s}  mean={g["boltz_plddt"].mean():.3f}  '
          f'sd={g["boltz_plddt"].std():.3f}  '
          f'esm_mean={g["esm_plddt"].mean():.3f}')

---
## Cell 8 — Figures

Fig M: cross-run violin — all 5 buckets (concordance, native_ala, scrambled_na, free_design, native_ala_free)  
Fig N: scrambled_na per-bar strip with ESMFold overlay  
Fig O: ESMFold vs Boltz pLDDT scatter for scrambled_na (are they consistent?)

In [ ]:
DARK_BG = '#0e0e0e'
COLORS  = {
    'concordance':   '#00d4ff',
    'native_ala':    '#ff9500',
    'scrambled_na':  '#888888',
    'free_design':   '#a855f7',
    'native_ala_free': '#39d353',
    'scrambled_naf': '#bbbbbb',
}

# Load historical results for comparison
hist_rows = []
for csv_path, label in [(V2_CSV, 'v2'), (V3_CSV, 'v3')]:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        hist_rows.append(df)
        print(f'Loaded {len(df)} rows from {csv_path.name}')
    else:
        print(f'[SKIP] {csv_path.name} not found')

hist_df = pd.concat(hist_rows, ignore_index=True) if hist_rows else pd.DataFrame()

# Combine all data
all_df = pd.concat([hist_df, results], ignore_index=True)
bucket_order = ['concordance', 'native_ala', 'scrambled_na', 'free_design', 'scrambled_naf', 'native_ala_free']
present = [b for b in bucket_order if b in all_df['bucket'].unique()]

# ── Fig M: cross-run violin ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

for xi, bucket in enumerate(present):
    vals = all_df[all_df['bucket'] == bucket]['boltz_plddt'].dropna().values
    if len(vals) == 0:
        continue
    parts = ax.violinplot([vals], positions=[xi], widths=0.7, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(COLORS.get(bucket, '#888'))
        pc.set_alpha(0.75)
    parts['cmedians'].set_colors(COLORS.get(bucket, '#888'))
    for k in ['cbars', 'cmins', 'cmaxes']:
        parts[k].set_colors('#555')
    mean_v = vals.mean()
    ax.text(xi, mean_v + 0.02, f'{mean_v:.3f}', ha='center', va='bottom',
            color='white', fontsize=8)

ax.set_xticks(range(len(present)))
ax.set_xticklabels(present, rotation=20, ha='right', color='white', fontsize=10)
ax.set_ylabel('Boltz-2 pLDDT', color='white')
ax.set_title('Rap Snacks — Boltz pLDDT across all design buckets\n'
             'scrambled_na = composition baseline (same AA, random order)',
             color='white', fontsize=12)
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_edgecolor('#333')
plt.tight_layout()
fig_m = DRIVE_FIGS / 'fig_v4_violin_all_buckets.png'
plt.savefig(fig_m, dpi=150, facecolor=DARK_BG, bbox_inches='tight')
plt.close()
print(f'Saved Fig M → {fig_m.name}')

# ── Fig N: scrambled_na per-bar strip ────────────────────────────────────────
bar_ids_sorted = sorted(results['bar_id'].unique())
fig, ax = plt.subplots(figsize=(14, 5), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

for xi, bar_id in enumerate(bar_ids_sorted):
    g = results[results['bar_id'] == bar_id]
    ax.scatter([xi] * len(g), g['boltz_plddt'],
               color=COLORS['scrambled_na'], alpha=0.75, s=40, zorder=3)
    ax.scatter([xi], [g['boltz_plddt'].mean()],
               color='white', s=60, zorder=4, marker='_', linewidths=2)
    # ESMFold overlay
    ax.scatter([xi], [g['esm_plddt'].mean()],
               color='#ef4444', s=40, zorder=4, marker='D', alpha=0.8)

ax.set_xticks(range(len(bar_ids_sorted)))
ax.set_xticklabels(bar_ids_sorted, rotation=45, ha='right', color='white', fontsize=8)
ax.set_ylabel('pLDDT', color='white')
ax.set_title('scrambled_na per bar — Boltz-2 (grey) vs ESMFold (red diamond)',
             color='white', fontsize=11)
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_edgecolor('#333')
ax.set_ylim(0, 1)
plt.tight_layout()
fig_n = DRIVE_FIGS / 'fig_v4_sc_na_per_bar.png'
plt.savefig(fig_n, dpi=150, facecolor=DARK_BG, bbox_inches='tight')
plt.close()
print(f'Saved Fig N → {fig_n.name}')

# ── Fig O: ESMFold vs Boltz scatter for scrambled_na ────────────────────────
fig, ax = plt.subplots(figsize=(7, 6), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)
ax.scatter(results['esm_plddt'], results['boltz_plddt'],
           color=COLORS['scrambled_na'], alpha=0.7, s=40)
ax.plot([0, 1], [0, 1], '--', color='#555', lw=1)
r_val = np.corrcoef(results['esm_plddt'].dropna(), results['boltz_plddt'].dropna())[0, 1]
ax.text(0.05, 0.95, f'r = {r_val:.3f}', transform=ax.transAxes,
        color='white', fontsize=11, va='top')
ax.set_xlabel('ESMFold pLDDT', color='white')
ax.set_ylabel('Boltz-2 pLDDT', color='white')
ax.set_title('scrambled_na — ESMFold vs Boltz-2 pLDDT\n(do both methods agree on composition baseline?)',
             color='white', fontsize=11)
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_edgecolor('#333')
plt.tight_layout()
fig_o = DRIVE_FIGS / 'fig_v4_esm_vs_boltz_sc_na.png'
plt.savefig(fig_o, dpi=150, facecolor=DARK_BG, bbox_inches='tight')
plt.close()
print(f'Saved Fig O → {fig_o.name}')

---
## Cell 9 — Summary

In [ ]:
print('=== Boltz Validation v4 Summary — scrambled_na ===')
print(f'Total predictions: {len(results)}')
print(f'  scrambled_na  n={len(results):4d}  '
      f'mean pLDDT={results["boltz_plddt"].mean():.3f}  '
      f'sd={results["boltz_plddt"].std():.3f}')

print('\n--- Per-bar scrambled_na Boltz pLDDT ---')
print(f'{"bar":8s}  {"n":>3s}  {"boltz_mean":>10s}  {"boltz_sd":>8s}  {"esm_mean":>9s}')
for bar_id, g in results.groupby('bar_id'):
    print(f'{bar_id:8s}  {len(g):3d}  '
          f'{g["boltz_plddt"].mean():10.3f}  '
          f'{g["boltz_plddt"].std():8.3f}  '
          f'{g["esm_plddt"].mean():9.3f}')

print('\n--- Cross-bucket comparison (Boltz pLDDT) ---')
ref = {
    'concordance':     0.441,
    'native_ala':      0.543,
    'free_design':     0.643,
    'native_ala_free': 0.806,
}
sc_mean = results['boltz_plddt'].mean()
print(f'  concordance     0.441')
print(f'  native_ala      0.543')
print(f'  scrambled_na    {sc_mean:.3f}  ← this run')
print(f'  free_design     0.643')
print(f'  native_ala_free 0.806')
print(f'\n  Gap (scrambled_na → native_ala): {sc_mean - 0.543:+.3f}')
print(f'  Gap (scrambled_na → free_design): {0.643 - sc_mean:+.3f}')

print('\nDone. Download results CSV and figures from Drive, then commit.')
print(f'  {RESULTS_CSV}')